# Movie Recommender System - Approach

**Team Members:** Bekithemba Nkomo, Peter Mangoro, Masheia Dzimba.

**Assignment:** Building a Hybrid Movie Recommender in Neo4j  

---

## How We Will Tackle This Problem

Our strategy is to build the recommender system in **progressive layers**, starting simple and adding complexity.

## Data and Graph Design

We will start from the three CSVs:

- `users.csv`: basic demographics (`userId`, `age`, `gender`, `occupation`).
- `movies.csv`: metadata (`title`, `year`, `genre1`, `genre2`, `director`, `avgRating`).
- `ratings.csv`: user–movie interactions (`rating` 1–5, `timestamp`).

Our **graph model**:

- Nodes:
  - `(:User {userId, name, age, gender, occupation})`
  - `(:Movie {movieId, title, year, genre1, genre2, director, avgRating})`
  - `(:Genre {name})`
  - `(:Director {name})`
- Relationships:
  - `(:User)-[:RATED {rating, timestamp}]->(:Movie)`
  - `(:Movie)-[:IN_GENRE]->(:Genre)`
  - `(:Movie)-[:DIRECTED_BY]->(:Director)`

This gives us a **bipartite core** (User–Movie) plus a **content layer** (Genre, Director) to support both collaborative and hybrid methods.

---

## Methodological Plan

### Phase 1: Foundation (Load & Enrich)

We will start by loading the three CSV files (users, movies, ratings) into Neo4j to create a basic bipartite graph: Users connected to Movies through `RATED` relationships. Then we will **enrich** this structure by extracting Genre and Director as first-class nodes.

Why materialize Genre/Director nodes? Because it enables patterns like: "find movies that share a genre with movies this user rated highly." As properties, we would need property matching; as nodes, we can traverse the graph naturally.

In our implementation we will make three concrete improvements to the baseline Step 1–2 scripts:

- **Use `MERGE` instead of `CREATE`** for `User`, `Movie`, and `RATED` so the import is idempotent and safe to re-run.
- **Apply `trim()` and proper type casting** (e.g. `trim(row.userId)`, `toInteger(row.age)`, `toFloat(row.rating)`, `datetime(row.timestamp)`) so identifiers and types are clean.
- **Add uniqueness constraints** for enrichment nodes (`Genre.name` and `Director.name`) to prevent duplicate content nodes.

**Reasoning:** Starting from a clean, verified graph reduces noise in all later similarity and recommendation steps.

---

### Phase 2: Exploration (EDA Queries)

Before computing anything, we need to understand what we are working with. The 8 EDA queries will reveal:

- **Sparsity**: only 100 ratings across 20×25 possible user-movie pairs.
- **Popularity bias / long-tail**: whether a few movies receive most ratings.
- **Rating inflation**: whether most ratings are 4–5.
- **Genre clustering**: whether preferences segment clearly by genre.

These patterns directly inform our algorithm choices (e.g., if ratings are inflated, magnitude-aware similarity may behave differently).

---

### Phase 3: Deeper Analytical Questions (2 of 4)

We will choose **two** of the analytical questions and treat each as a mini-analysis:

- Define the **metric** precisely.
- Implement the metric in Cypher (and, if needed, simple Pandas post-processing).
- Interpret results in terms of similarity validity, genre profiles, long-tail/hidden gems, and director–genre structure.

**Reasoning:** These deeper questions guide similarity and hybrid scoring design.

---

### Phase 4: GDS Similarity (Jaccard vs FastRP + kNN)

We will compute user-user similarity two ways:

- **Jaccard (unweighted):** Treats ratings as binary (rated vs. not rated). Simple and interpretable, but ignores rating magnitude.
- **FastRP + kNN (weighted):** Uses rating-weighted structure to embed nodes, then computes cosine-based user similarity in embedding space.

We will write both similarity types back to the graph as relationships (`SIMILAR_TASTE`, `KNN_SIMILAR`) so we can reuse them for recommendations.

---

### Phase 5: Recommendations (Collaborative + Hybrid)

We will generate personalised recommendations using the user-user similarity relationships.

- **Collaborative filtering:** Recommend movies that similar users rated highly (e.g. rating ≥ 4), excluding movies the target has already rated.
- **Hybrid recommendations:** Boost collaborative candidates that share **genre** and/or **director** with movies the user has rated ≥ 4.

We will implement a simple, interpretable hybrid mix (e.g. **70% collaborative signal / 30% content boost**) and compare hybrid vs. pure collaborative results.

---

### Phase 6: Community Detection and Simple Evaluation

We will:

1. Construct a **User–User similarity graph** for community detection.
2. Run a community detection algorithm (e.g. **Louvain**) and characterize the 3 largest communities by dominant genres, average age, and occupation distribution.
3. Perform a **simple hold-out evaluation** by temporarily hiding two ratings for one user, regenerating recommendations, and computing **precision@K** (K between 3 and 10).

**Reasoning:** Communities reveal clusters of taste; hold-out evaluation forces explicit measurement (with clear limitations on such a small dataset).

---

### Phase 7: Extension Exercises

After the core pipeline is working, we will address the four extension tasks:

1. **Similarity Cutoff Sensitivity:** Re-run Node Similarity with different cutoffs, track the number of `SIMILAR_TASTE` edges created, and observe how recommendation coverage/quality changes.
2. **Cold-Start Problem:** Introduce a new user with a single high rating; compare collaborative vs. hybrid behavior; implement a concrete cold-start fallback strategy.
3. **Algorithm Comparison:** For one user, compare top 5 recommendations from (a) Jaccard-based CF, (b) kNN/cosine-based CF, and (c) hybrid CF + content.
4. **Director Affinity Boost:** Modify the hybrid query to reward movies by directors the user has rated highly; compare before/after lists and discuss impact.

**Reasoning:** These exercises stress-test the recommender under parameter changes, sparse users, and different modeling choices.

---

## Anticipated Challenges

### Challenge 1: Graph Sparsity
**The problem:** 100 ratings across 500 possible user-movie pairs = 80% empty matrix. Most user pairs will not have any co-rated movies.

**Why this matters:**
- Jaccard similarity will be 0 for most user pairs (no overlap)
- Community detection might produce many singleton clusters
- Recommendations might be limited to only the most popular movies

**How we will handle it:**
- Use **lower similarity cutoffs** (0.2-0.3 instead of 0.5) to capture weak but meaningful connections
- Rely on **content signals** (genre, director) to fill gaps where collaborative signals fail
- **Hybrid scoring** ensures that even if collaborative filtering is weak, content matching can still recommend relevant items
- Be honest in evaluation: acknowledge that results are exploratory given the small sample

---

### Challenge 2: Jaccard vs Cosine - Which to Use?
**The dilemma:** Jaccard and cosine capture different things, and it is not obvious which is better.

**Example where they diverge:**
- User A rates {M1: 5, M2: 5, M3: 5}
- User B rates {M1: 1, M2: 1, M3: 1}
- Jaccard similarity = 1.0 (perfect overlap!)
- Cosine similarity = -1.0 (complete disagreement!)

Jaccard finds users who rated the same movies. Cosine finds users who rated them similarly.

**How we will handle it:**
- Implement **both** and compare results explicitly (Extension 3)
- Use Jaccard for initial exploration (simpler, easier to interpret)
- Use cosine/FastRP for final recommendations (captures agreement, not just overlap)
- Document where rankings diverge and explain why

---

### Challenge 3: Evaluation Is a Toy Test
**The problem:** Hold-out testing on one user with 2 hidden ratings is not statistically meaningful.

**Limitations we acknowledge upfront:**
- Cannot compute confidence intervals or significance tests
- Results could be luck (which 2 ratings were hidden)
- No baseline comparison (is our system better than "just recommend popular movies"?)

**How we will handle it:**
- Run hold-out on **multiple users** (3-5) to see if results generalize
- **Qualitative assessment:** Manually inspect recommendations, do they make sense given the user taste?
- **Explicitly state limitations** in the write-up (this is proof-of-concept, not production validation)
- **Discuss what production would look like:** k-fold cross-validation, A/B testing, precision@K, recall@K, NDCG, comparison to baselines

---

### Challenge 4: Cold-Start Problem
**The problem:** New users with 0-1 ratings have no collaborative filtering signal.

**Example (Extension 2):** We will create user U021 who has only rated "The Matrix" (5 stars). Jaccard and kNN will fail, no similar users can be found.

**How we will handle it:**
- **Content-based fallback:** Recommend movies with same genre/director as "The Matrix" (Sci-Fi/Action, Wachowski Sisters)
- **Popularity baseline:** Default to highest-rated movies overall
- **Demographic init:** Recommend popular movies within the user age group or occupation

Cold-start is where hybrid systems shine. Demonstrating graceful degradation (content-based works when collaborative fails) shows robustness.

---

### Challenge 5: Dual-Genre Weighting
**The problem:** Movies have up to 2 genres. How do we score genre matches?

**Ambiguous scenarios:**
- User loves Sci-Fi. Candidate A is {Sci-Fi, Action}. Candidate B is {Sci-Fi, Horror}. Same score?
- User rated 3 pure Sci-Fi movies highly. Should {Sci-Fi, Thriller} get full credit or partial?

**How we will handle it:**
- Treat both genre1 and genre2 equally: either match counts as a hit
- Normalize: 1 match = +0.5 boost, 2 matches = +1.0 boost
- **Acknowledge limitation:** Dual genres blur content signals; production systems would use TF-IDF on multi-hot genre vectors

---

### Challenge 6: GDS Projection Design
**The problem:** Different algorithms need different graphs.

**Examples:**
- **Node Similarity:** Needs a bipartite User–Movie graph
- **FastRP + kNN:** Needs a weighted bipartite graph (rating as relationship weight)
- **Community Detection:** Needs a monopartite User–User similarity graph

**How we will handle it:**
- Create **task-specific projections** with descriptive names (matching our implementation notebook):
  - `user_movie_unweighted` (for Jaccard Node Similarity)
  - `user_movie_weighted` (for FastRP + kNN)
  - `user_user_similarity` (for Louvain on user–user similarity edges)
- **Document each projection design:** which nodes, which relationships, properties loaded, directed vs. undirected, and why.
- Use `gds.graph.drop()` to clean up between tasks.

---

### Challenge 7: Parameter Tuning Without Ground Truth
**The problem:** How do we know if `similarityCutoff=0.3` is better than `0.5`? Or `embeddingDimension=64` vs `128`?

**How we will handle it:**
- **Sensitivity analysis** (Extension 1): Test cutoffs {0.1, 0.3, 0.5} and report:
  - Number of similarity edges created (graph density)
  - Number of recommendations generated (coverage)
  - Qualitative assessment (do results look reasonable?)
- **Use literature defaults:** FastRP paper suggests embeddingDimension=64-128 for small graphs
- **Justify in write-up:** Explain every parameter choice, even if it is an educated guess

---

## Approach Rationale

### Connection to MMDS Chapter 9
The textbook models recommendations as a sparse utility matrix (users × items). Neo4j represents this as a **bipartite graph**, relationships ARE the non-blank matrix entries. This is not just a different data structure; it is a better conceptual model:

- **Collaborative filtering** = graph traversal ("find users who rated similar movies")
- **Content-based filtering** = metadata integration (Genre/Director nodes)
- **Similarity computation** = GDS algorithms (Jaccard, FastRP, kNN)

The chapter distinguishes Jaccard (binary overlap) from cosine (magnitude-aware). Our approach implements both so we can compare them empirically.

### Hybrid > Pure Collaborative Filtering
Chapter 9 notes that real systems combine collaborative and content-based approaches. Our 70/30 hybrid balances:
- **Proven effectiveness** (collaborative filtering works)
- **Diversity & cold-start resilience** (content signals fill gaps)
- **Explainability** ("because you liked other Nolan films" > "users similar to you liked this")

### Progressive Complexity
We are starting simple (Jaccard, pure collaborative) and layering sophistication (FastRP, hybrid, community detection). This mirrors real-world development: build a baseline, measure it, improve iteratively.

---

## Next Steps

Once the approach is approved, we will:

1. **Run graph enrichment** (create Genre/Director nodes)
2. **Complete all 8 EDA queries** with commentary
3. **Implement 2 analytical questions** (Genre profiles including Long-tail)
4. **Execute all 5 GDS tasks** (Jaccard, FastRP+kNN, recommendations, hybrid, community detection)
5. **Complete all 4 extensions**
---

## References

**Course Materials:**
- Leskovec, J., Rajaraman, A., & Ullman, J. D. (2020). *Mining of Massive Datasets*, Chapter 9: Recommendation Systems.

**Neo4j Documentation:**
- Neo4j Graph Data Science Library. https://neo4j.com/docs/graph-data-science/current/
- Node Similarity (Jaccard). https://neo4j.com/docs/graph-data-science/current/algorithms/node-similarity/
- k-Nearest Neighbors. https://neo4j.com/docs/graph-data-science/current/algorithms/knn/
- FastRP. https://neo4j.com/docs/graph-data-science/current/machine-learning/node-embeddings/fastrp/
- Louvain Community Detection. https://neo4j.com/docs/graph-data-science/current/algorithms/louvain/

**Academic Literature:**
- Koren, Y., Bell, R., & Volinsky, C. (2009). Matrix Factorization Techniques for Recommender Systems. *IEEE Computer*, 42(8), 30-37.
- Herlocker, J. L., Konstan, J. A., Terveen, L. G., & Riedl, J. T. (2004). Evaluating Collaborative Filtering Recommender Systems. *ACM Transactions on Information Systems*, 22(1), 5-53.

---